# Empirical Bayes rate smoothing

Compare crude rates with Marshall's global and local moment shrinkage estimators. Synthetic counts are used to expose the mechanics, not to represent a real health dataset.

In [ ]:
import sys
from pathlib import Path
repo_root = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "implementations/python/empirical-bayes-rates").is_dir())
sys.path.insert(0, str(repo_root / "implementations/python/empirical-bayes-rates"))
from empirical_bayes import global_empirical_bayes, local_empirical_bayes
import numpy as np
import matplotlib.pyplot as plt


## 1. Global shrinkage

The global estimator shrinks toward the pooled rate. It does not use area adjacency.

In [ ]:
counts = [1, 2, 4, 18, 22, 3, 1, 16]
exposure = [100, 120, 90, 1000, 1100, 100, 80, 900]
global_result = global_empirical_bayes(counts, exposure)
print('pooled rate:', global_result['overall_rate'])
print('between-area variance:', global_result['between_area_variance'])
print('crude:', np.round(global_result['crude_rates'], 4))
print('smoothed:', np.round(global_result['estimates'], 4))


In [ ]:
x = np.arange(len(counts))
plt.figure(figsize=(9, 4))
plt.plot(x, global_result['crude_rates'], 'o-', label='Crude rate')
plt.plot(x, global_result['estimates'], 's-', label='Global EB rate')
plt.axhline(global_result['overall_rate'], linestyle='--', label='Pooled rate')
plt.xlabel('Area index'); plt.ylabel('Rate per exposure unit'); plt.legend(); plt.show()


## 2. Local shrinkage

The neighbourhood graph is explicit input. This example uses adjacent groups for teaching; a real analysis should derive neighbourhoods from a documented contiguity or distance rule.

In [ ]:
neighborhoods = [[0, 1, 2], [0, 1, 2, 3], [1, 2, 3, 4], [2, 3, 4, 5],
                 [3, 4, 5, 6], [4, 5, 6, 7], [5, 6, 7], [5, 6, 7]]
local_result = local_empirical_bayes(counts, exposure, neighborhoods)
print('local rates:', np.round(local_result['estimates'], 4))


## 3. Interpret the difference

Global estimates are invariant to map layout. Local estimates depend on neighbourhood membership and assume nearby areas can reasonably share a local prior. Neither estimator is a cluster significance test. A production disease map may need age standardisation, covariates, overdispersion, temporal structure and a CAR/BYM model.

## Questions
1. Which crude rates shrink most, and how does exposure explain this?
2. Why does the global estimator ignore map geometry?
3. How does changing one area's neighbourhood alter local estimates?
4. Why is a smoothed choropleth not a cluster significance test?
